# 1. Original SimEvents results and validation of the Python model

This notebook (i) reviews the results of the original MATLAB/SimEvents model of the evacuation of Building G (UFV) and (ii) validates an independent **Python/SimPy re-implementation** of that model against them.

**Model in one paragraph.** The building is a network of FIFO queues. Each door, corridor and stair is a *server* with a capacity (people that can traverse it simultaneously) and a traversal time $t = d / (v_k f)$, where $v_k$ is the occupant's walking speed and $f=\alpha_k$ on stairs, 1 otherwise. Each occupant waits a random pre-movement time, walks to the first element of its assigned route and then queues element by element until it reaches an exit. Occupant profiles (speed, stair factor, pre-movement time) and occupancy per scenario come from the project report.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evacuation import *
from evacuation import config
from evacuation.plots import use_style, BLUE, ORANGE, AQUA, INK, INK_2, MUTED

use_style()
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
RES = Path("../results"); RES.mkdir(exist_ok=True)
pd.set_option("display.precision", 1)

## 1.1 Original SimEvents results

The SimEvents export contains, for each scenario, the time at which every occupant left through each of the 8 exits (one replication with seed `rng(42)`).

In [2]:
scen = ["E1", "E2", "E3", "E4"]
tmax_se = pd.Series({s: simevents_summary(s)["clearance_s"].max() for s in scen}, name="SimEvents T_max (s)")
people_se = pd.Series({s: int(simevents_summary(s)["evacuated"].sum()) for s in scen}, name="occupants")
pd.concat([people_se, tmax_se], axis=1).assign(**{"reduction vs E1": lambda d: (1 - d.iloc[:, 1] / d.iloc[0, 1]).map("{:.1%}".format)})

,occupants,SimEvents T_max (s),reduction vs E1
E1,823,365.5,0.0%
E2,647,329.8,9.8%
E3,366,265.0,27.5%
E4,288,193.7,47.0%


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, s in zip(axes.flat, scen):
    df = load_simevents(s)
    for e in config.EXITS:
        d = df[df.exit == e]
        if d.empty:
            continue
        highlight = {"stair_ext_left": ORANGE, "stair_ext_right": BLUE}.get(e)
        label = config.EXIT_LABELS[e] if highlight else ("Other exits" if e == "pm1_left" else None)
        ax.step(d.time_s, d.cumulative_evacuated, where="post", label=label,
                color=highlight or MUTED, lw=2.2 if highlight else 1.2, zorder=3 if highlight else 2)
    ax.axvline(tmax_se[s], color=INK, lw=0.8, ls="--")
    ax.set_title(f"{s} - {config.SCENARIOS[s]['label']}   T_max = {tmax_se[s]:.0f} s")
for ax in axes[1]: ax.set_xlabel("Time since alarm (s)")
for ax in axes[:, 0]: ax.set_ylabel("Cumulative evacuated")
handles, labels = axes.flat[0].get_legend_handles_labels()
order = [labels.index(l) for l in ["External stair (left)", "External stair (right)", "Other exits"]]
fig.legend([handles[i] for i in order], [labels[i] for i in order], loc="upper right", ncol=3, bbox_to_anchor=(0.99, 1.0))
fig.suptitle("SimEvents: cumulative evacuees per exit", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(FIG / "01_simevents_cumulative.png"); plt.show()

The external stairs (left in particular) are the last exits to clear in every scenario: they collect the flow from the first-floor walkway **and** from the second floor.

## 1.2 Calibration

The report documents occupant profiles, occupancy and routes, but not the lengths and capacities of doors and stairs. Three parameters were calibrated on scenario **E1 only**, minimising the RMSE between the mean simulated clearance time of each exit (20 replications, common random numbers) and the SimEvents value (Nelder-Mead). Set `RUN_CALIBRATION = True` to repeat it (about 1 min).

In [4]:
from evacuation.experiments import calibrate, calibration_loss
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    cal = calibrate()
    print(cal["params"], "RMSE:", round(cal["rmse_s"], 2), "s")
x = [config.DOOR_LENGTH, config.LANDING_DOOR_LENGTH, config.EXT_STAIR_LENGTH]
print(f"door length = {x[0]} m, landing door length = {x[1]} m, external stair length = {x[2]} m")
print(f"calibration RMSE on E1 exit clearance times: {calibration_loss(x):.1f} s")

door length = 1.98 m, landing door length = 2.11 m, external stair length = 8.64 m


calibration RMSE on E1 exit clearance times: 5.2 s


## 1.3 Validation: total evacuation time

SimEvents provides a single replication per scenario, so the right comparison is whether that value is a plausible draw from the Python model's distribution. E2-E4 are **out of sample** (not used for calibration).

In [5]:
N_VAL = 300
val = {s: monte_carlo(s, n_reps=N_VAL) for s in scen}
rows = []
for s in scen:
    t = val[s]["runs"].t_max
    rows.append({"scenario": s, "SimEvents": tmax_se[s], "Python mean": t.mean(),
                 "P05": t.quantile(.05), "P95": t.quantile(.95),
                 "SimEvents percentile": (t < tmax_se[s]).mean() * 100,
                 "error of mean": f"{t.mean() / tmax_se[s] - 1:+.1%}"})
validation = pd.DataFrame(rows).set_index("scenario"); validation

,SimEvents,Python mean,P05,P95,SimEvents percentile,error of mean
scenario,,,,,,
E1,365.5,366.3,342.8,389.2,47.3,+0.2%
E2,329.8,340.3,325.7,354.8,11.7,+3.2%
E3,265.0,264.4,254.3,275.5,54.0,-0.2%
E4,193.7,200.0,193.0,208.0,5.3,+3.2%


In [6]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, s in zip(axes, scen):
    t = val[s]["runs"].t_max
    ax.hist(t, bins=25, color=BLUE, edgecolor="white", linewidth=0.6)
    ax.axvline(tmax_se[s], color=ORANGE, lw=2)
    ax.set_title(f"{s}   SimEvents = {tmax_se[s]:.0f} s"); ax.set_xlabel("T_max (s)"); ax.set_yticks([])
fig.suptitle(f"Python model ({N_VAL} replications, blue) vs the single SimEvents run (orange line); E2-E4 out of sample",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(FIG / "01_validation_tmax.png"); plt.show()

## 1.4 Validation: clearance time of every exit (E1)

In [7]:
r = val["E1"]["runs"]; se = simevents_summary("E1")
labels = [config.EXIT_LABELS[e] for e in config.EXITS]
mean = np.array([r[f"clear_{e}"].mean() for e in config.EXITS])
lo = np.array([r[f"clear_{e}"].quantile(.05) for e in config.EXITS])
hi = np.array([r[f"clear_{e}"].quantile(.95) for e in config.EXITS])
y = np.arange(len(labels))[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hlines(y, lo, hi, color=BLUE, lw=6, alpha=0.35, label="Python P05-P95")
ax.plot(mean, y, "o", color=BLUE, ms=8, label="Python mean")
ax.plot(se.clearance_s.values, y, "D", color=ORANGE, ms=8, mec="white", mew=1.5, label="SimEvents")
ax.set_yticks(y, labels); ax.set_xlabel("Clearance time of the exit (s)")
ax.set_title("E1 (peak hours): clearance time per exit"); ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(FIG / "01_validation_exits_E1.png"); plt.show()

The only visible gap is the P0 centre door (~13 s later in Python). That exit serves only 60 people, so its clearance time depends on details of the original room-to-door distances that the report does not give; it never determines the total evacuation time.

**Conclusion.** With three calibrated geometric parameters, the Python model reproduces the SimEvents clearance time of every exit in E1 and its total evacuation time in the three out-of-sample scenarios: the SimEvents value lies inside the Python 5-95 % band in all four scenarios. The model also reproduces the per-exit head counts (they follow from occupancy and routing).